In [1]:
#Loan Env Variable
import json
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# Create an API Client 
from anthropic import Anthropic

client = Anthropic()
model = "claude-opus-4-5-20251101"

In [3]:
# Make a request

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system = None, stop_sequences=None):
    params = {
        "model" : model,
        "max_tokens" : 1000,
        "messages" : messages,
        # "temperature" : temperature,
    }
    if system:
        params["system"] = system
    
    if stop_sequences:
        params["stop_sequences"] = stop_sequences


    message = client.messages.create(**params)
    return message.content[0].text


In [4]:
#Make a starting list of messages
messages = []

#Add in the initailise user quastion of "Define Quantum Computing in one sentence"
add_user_message(messages, "Define Quantum Computing in one sentence")

# Pass the list of the message into 'chat' to get an answer
answer = chat(messages)

#take the answer and add it as assistant message into our list 
add_assistant_message(messages, answer)

#Add in the user's follow-up quastion 
add_user_message(messages, "Write another sentence")

answer = chat(messages)
# answer

In [5]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result."""
    prompt = f"""
Please solve the following task:
{test_case["task"]}
* Respond only with Python, JSON, or a plain Regex
*Do not add any comments or commentary or explanation 
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [6]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    OriginalTask: 
    <task>
    {test_case['task']}
    </task>
    Solution to Evaluate: 
    <solution>
    {output}    
    </solution>

    Criteria you should use to evaluate the solution:
    <criteria>
    {test_case['solution_criteria']}
    </criteria>

    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10


    Response with JSON. Keep your response concise and direct.
    Example response shape:
    {{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
    }}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [7]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [8]:
def run_test_cases(test_cases):
    """Calls run_prompt, then grades the output."""
    output = run_prompt(test_cases)

    # TODO - Gradient
    model_grade = grade_by_model(test_cases, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_cases)
    score = (model_score + syntax_score) / 2

    return{
        "output": output,
        "test_cases": test_cases,
        "score": score,
        "reasoning": reasoning
    }

In [9]:
from statistics import mean
def run_eval(dataset):
    """Load the dataset and call run_test_case with each case."""
    results = []
    for test_case in dataset:
        result = run_test_cases(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average Score: {average_score}")
    
    return results

In [10]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average Score: 9.5


In [11]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\ndef extract_s3_bucket_names(response: dict) -> list:\n    buckets = response.get(\"Buckets\", [])\n    bucket_names = [bucket.get(\"Name\") for bucket in buckets if bucket.get(\"Name\")]\n    return sorted(bucket_names)\n",
    "test_cases": {
      "task": "Write a Python function that extracts all S3 bucket names from a given AWS CLI response dictionary and returns them as a sorted list",
      "format": "python",
      "solution_criteria": "Function accepts a dict with 'Buckets' key containing list of bucket objects, extracts 'Name' from each, returns sorted list of strings"
    },
    "score": 9.0,
    "reasoning": "The solution correctly fulfills the core requirements: it extracts bucket names from the AWS CLI response dictionary structure, handles missing keys gracefully with .get(), filters out invalid names, and returns a sorted list. The implementation is concise and Pythonic. However, it lacks documentation, complete type hints (return type annotation),